# Fix: matriz Markov `pij_lndu` con fila `wetlands` rota

## Problema
En `apply_step0_verified.py` y `apply_step1_calibration.py` el filtro `'_to_wetlands' in c` se aplica a TODAS las columnas que terminan en `_to_wetlands`. Esto **también** captura la diagonal `pij_lndu_wetlands_to_wetlands`, dejando la fila completa de la cadena de Markov con suma = 0 en vez de 1.

El módulo AFOLU de SISEPUEDE itera esta matriz año por año. Con una fila no estocástica, entra a rutas de fallback (renormalización con división por cero, reasignaciones, logging) — ése es el cuello de botella observado.

## Fix
1. Zeroes solo las columnas `*_to_wetlands` **excluyendo** la diagonal (otras clases no pueden transicionar a wetlands → país árido, OK).
2. Pone `pij_lndu_wetlands_to_wetlands = 1.0` (wetlands queda como estado absorbente — su área no cambia).

## Alcance
Esta libreta parchea el archivo activo declarado en `config_files/config.yaml` (hoy: `sisepuede_raw_inputs_recalibrated_electricity_trns_improved.csv`). Hace backup con timestamp antes de sobreescribir.

In [9]:
from pathlib import Path
from datetime import datetime
import shutil
import pandas as pd
import numpy as np
import yaml

NB_DIR = Path.cwd()
REPO_ROOT = NB_DIR.parent
INPUT_DIR = REPO_ROOT / 'input_data'
CONFIG_PATH = NB_DIR / 'config_files' / 'config.yaml'

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

ACTIVE_CSV = INPUT_DIR / cfg['ssp_input_file_name']
print(f'Active input: {ACTIVE_CSV.name}')
print(f'Exists: {ACTIVE_CSV.exists()}')

Active input: sisepuede_raw_inputs_recalibrated_electricity_trns_improved.csv
Exists: True


## 1) Inspección ANTES del fix

In [10]:
df = pd.read_csv(ACTIVE_CSV)

wet_to_cols = sorted([c for c in df.columns if c.startswith('pij_lndu_') and c.endswith('_to_wetlands')])
wet_from_cols = sorted([c for c in df.columns if c.startswith('pij_lndu_wetlands_to_')])

print('Columnas "X -> wetlands" (in-flows hacia wetlands):')
for c in wet_to_cols:
    print(f'  {c}')
print(f'\nColumnas "wetlands -> Y" (fila wetlands de la matriz):')
for c in wet_from_cols:
    print(f'  {c}')

Columnas "X -> wetlands" (in-flows hacia wetlands):
  pij_lndu_croplands_to_wetlands
  pij_lndu_flooded_to_wetlands
  pij_lndu_forests_mangroves_to_wetlands
  pij_lndu_forests_primary_to_wetlands
  pij_lndu_forests_secondary_to_wetlands
  pij_lndu_grasslands_to_wetlands
  pij_lndu_other_to_wetlands
  pij_lndu_pastures_to_wetlands
  pij_lndu_settlements_to_wetlands
  pij_lndu_shrublands_to_wetlands
  pij_lndu_wetlands_to_wetlands

Columnas "wetlands -> Y" (fila wetlands de la matriz):
  pij_lndu_wetlands_to_croplands
  pij_lndu_wetlands_to_flooded
  pij_lndu_wetlands_to_forests_mangroves
  pij_lndu_wetlands_to_forests_primary
  pij_lndu_wetlands_to_forests_secondary
  pij_lndu_wetlands_to_grasslands
  pij_lndu_wetlands_to_other
  pij_lndu_wetlands_to_pastures
  pij_lndu_wetlands_to_settlements
  pij_lndu_wetlands_to_shrublands
  pij_lndu_wetlands_to_wetlands


In [11]:
row_sum_before = df[wet_from_cols].sum(axis=1)
diag_before = df['pij_lndu_wetlands_to_wetlands']

print('Fila wetlands de la matriz Markov (suma debería ser 1.0):')
print(f'  row_sum:   min={row_sum_before.min():.6f}  max={row_sum_before.max():.6f}')
print(f'  diagonal:  min={diag_before.min():.6f}  max={diag_before.max():.6f}')

broken = (row_sum_before < 0.999).any()
print(f'\n{"BROKEN" if broken else "OK"}: la fila {"no suma 1" if broken else "está sana"}.')

Fila wetlands de la matriz Markov (suma debería ser 1.0):
  row_sum:   min=0.000000  max=0.000000
  diagonal:  min=0.000000  max=0.000000

BROKEN: la fila no suma 1.


In [12]:
all_pij = [c for c in df.columns if c.startswith('pij_lndu_')]
from_states = sorted({c.replace('pij_lndu_', '').split('_to_')[0] for c in all_pij})

print('Suma de cada fila de la matriz Markov (todas deberían ser ~1.0):')
summary = []
for s in from_states:
    cols = [c for c in all_pij if c.startswith(f'pij_lndu_{s}_to_')]
    rs = df[cols].sum(axis=1)
    summary.append({'from_state': s, 'n_cols': len(cols), 'sum_min': rs.min(), 'sum_max': rs.max()})
summary_df = pd.DataFrame(summary)
summary_df['ok'] = (summary_df.sum_min > 0.999) & (summary_df.sum_max < 1.001)
summary_df

Suma de cada fila de la matriz Markov (todas deberían ser ~1.0):


,from_state,n_cols,sum_min,sum_max,ok
0,croplands,11,0.999994,1.000000,True
1,flooded,11,0.999875,1.000000,True
2,forests_mangroves,11,1.000000,1.000000,True
3,forests_primary,11,0.999955,1.000000,True
4,forests_secondary,11,0.999993,0.999999,True
5,grasslands,11,0.999979,1.000000,True
6,other,11,0.999998,1.000000,True
7,pastures,11,0.999978,1.000000,True
8,settlements,11,1.000000,1.000000,True
9,shrublands,11,0.999992,1.000000,True


## 2) Aplicar el fix

In [13]:
DIAG = 'pij_lndu_wetlands_to_wetlands'
off_diag_to_wetlands = [c for c in wet_to_cols if c != DIAG]

df_fixed = df.copy()
for c in off_diag_to_wetlands:
    df_fixed[c] = 0.0
df_fixed[DIAG] = 1.0

print(f'Zeroed {len(off_diag_to_wetlands)} columnas "X -> wetlands" (excluida la diagonal):')
for c in off_diag_to_wetlands:
    print(f'  {c}')
print(f'\nSet {DIAG} = 1.0 (estado absorbente)')

Zeroed 10 columnas "X -> wetlands" (excluida la diagonal):
  pij_lndu_croplands_to_wetlands
  pij_lndu_flooded_to_wetlands
  pij_lndu_forests_mangroves_to_wetlands
  pij_lndu_forests_primary_to_wetlands
  pij_lndu_forests_secondary_to_wetlands
  pij_lndu_grasslands_to_wetlands
  pij_lndu_other_to_wetlands
  pij_lndu_pastures_to_wetlands
  pij_lndu_settlements_to_wetlands
  pij_lndu_shrublands_to_wetlands

Set pij_lndu_wetlands_to_wetlands = 1.0 (estado absorbente)


## 3) Verificación DESPUÉS del fix

In [14]:
row_sum_after = df_fixed[wet_from_cols].sum(axis=1)
print(f'Fila wetlands después del fix:')
print(f'  row_sum:  min={row_sum_after.min():.6f}  max={row_sum_after.max():.6f}')

print('\nSuma de TODAS las filas Markov después del fix:')
summary_after = []
for s in from_states:
    cols = [c for c in all_pij if c.startswith(f'pij_lndu_{s}_to_')]
    rs = df_fixed[cols].sum(axis=1)
    summary_after.append({'from_state': s, 'sum_min': rs.min(), 'sum_max': rs.max()})
summary_after_df = pd.DataFrame(summary_after)
summary_after_df['ok'] = (summary_after_df.sum_min > 0.999) & (summary_after_df.sum_max < 1.001)
summary_after_df

Fila wetlands después del fix:
  row_sum:  min=1.000000  max=1.000000

Suma de TODAS las filas Markov después del fix:


,from_state,sum_min,sum_max,ok
0,croplands,0.999994,1.000000,True
1,flooded,0.999875,1.000000,True
2,forests_mangroves,1.000000,1.000000,True
3,forests_primary,0.999955,1.000000,True
4,forests_secondary,0.999993,0.999999,True
5,grasslands,0.999979,1.000000,True
6,other,0.999998,1.000000,True
7,pastures,0.999978,1.000000,True
8,settlements,1.000000,1.000000,True
9,shrublands,0.999992,1.000000,True


In [15]:
all_ok = summary_after_df['ok'].all()
assert all_ok, 'Hay filas Markov que aún no suman 1 después del fix'
print('Todas las filas Markov pij_lndu_* suman 1.0 — matriz estocástica OK.')

Todas las filas Markov pij_lndu_* suman 1.0 — matriz estocástica OK.


## 4) Guardar (con backup)

Cambiar `APPLY = True` y re-ejecutar la celda para sobreescribir el input activo. Se crea un `.bak_<timestamp>` antes.

In [16]:
APPLY = True

if APPLY:
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    backup = ACTIVE_CSV.with_suffix(f'.csv.bak_{ts}')
    shutil.copy2(ACTIVE_CSV, backup)
    df_fixed.to_csv(ACTIVE_CSV, index=False)
    print(f'Backup -> {backup.name}')
    print(f'Wrote  -> {ACTIVE_CSV.name}')

    reloaded = pd.read_csv(ACTIVE_CSV, usecols=wet_from_cols)
    rs = reloaded.sum(axis=1)
    print(f'\nReload check — wetlands row sum: [{rs.min():.6f}, {rs.max():.6f}]')
else:
    print('APPLY=False — set APPLY=True para escribir el archivo.')

Backup -> sisepuede_raw_inputs_recalibrated_electricity_trns_improved.csv.bak_20260518_215541
Wrote  -> sisepuede_raw_inputs_recalibrated_electricity_trns_improved.csv

Reload check — wetlands row sum: [1.000000, 1.000000]
